# Copernicus Marine Service: Access Earth Observation Data
* Need to have an account on Copernicus Marine
* The ~/.netrc file contains the account credentials: your login and password for [Copernicus Marine](https://marine.copernicus.eu/)
```
$ more ~/.netrc
machine auth.marine.copernicus.eu
  login xxxxxxxxxx 
  password xxxxxxxxxxxxx
```

In [ ]:
import copernicusmarine
import hvplot.xarray

In [ ]:
# turn of some annoying warnings
import warnings
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

In [ ]:
dataset_id = 'SST_MED_SST_L3S_NRT_OBSERVATIONS_010_012_b'

In [ ]:
ds = copernicusmarine.open_dataset(dataset_id)

In [ ]:
%%time
da = ds['adjusted_sea_surface_temperature'].sel(time='2025-10-01 16:00', method='nearest').load() - 273.15

In [ ]:
da.hvplot(x='longitude', y='latitude', rasterize=True, geo=True, cmap='turbo', tiles='OSM')

## Local CMEMS file: Mediterranean Physics (thetao & bottomT)

**Dataset**: `cmems_mod_med_phy-temp_my_4.2km_P1D-m` (Mediterranean Multiyear Physics, 4.2 km, daily).

Contains **90 days (Jan–Mar 2026)** of:
- `thetao` — potential temperature at 1 m depth
- `bottomT` — bottom temperature

In [ ]:
import xarray as xr
import numpy as np
import matplotlib.pyplot as plt
import hvplot.xarray  # noqa: F401

local_nc = 'cmems_mod_med_phy-temp_my_4.2km_P1D-m_1776862674750.nc'
ds_med = xr.open_dataset(local_nc)
ds_med

In [ ]:
# Inspect available variables, dimensions, and time range
print('Variables :', list(ds_med.data_vars))
print('Dimensions:', dict(ds_med.sizes))
print('Time range:', str(ds_med.time.values[0])[:10], '->', str(ds_med.time.values[-1])[:10])

### 1. Spatial map — surface temperature (`thetao` at 1 m) on the first day

In [ ]:
# Select first time step; squeeze depth if present
sst0 = ds_med['thetao'].isel(time=0)
if 'depth' in sst0.dims:
    sst0 = sst0.isel(depth=0)

fig, ax = plt.subplots(figsize=(11, 4.5))
im = sst0.plot(ax=ax, cmap='turbo', cbar_kwargs={'label': '°C'})
ax.set_title(f"Surface potential temperature (thetao, 1 m) — {str(sst0.time.values)[:10]}")
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout(); plt.show()

### 2. Spatial map — bottom temperature (`bottomT`) on the first day

In [ ]:
bt0 = ds_med['bottomT'].isel(time=0)

fig, ax = plt.subplots(figsize=(11, 4.5))
bt0.plot(ax=ax, cmap='viridis', cbar_kwargs={'label': '°C'})
ax.set_title(f"Bottom temperature (bottomT) — {str(bt0.time.values)[:10]}")
ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout(); plt.show()

### 3. Basin-averaged daily time series (Jan–Mar 2026)

In [ ]:
thetao = ds_med['thetao']
if 'depth' in thetao.dims:
    thetao = thetao.isel(depth=0)

ts_surf = thetao.mean(dim=[d for d in thetao.dims if d != 'time'])
ts_bot  = ds_med['bottomT'].mean(dim=[d for d in ds_med['bottomT'].dims if d != 'time'])

fig, ax = plt.subplots(figsize=(11, 4))
ts_surf.plot(ax=ax, label='Surface (thetao @ 1 m)', color='tab:red')
ts_bot.plot(ax=ax, label='Bottom (bottomT)', color='tab:blue')
ax.set_ylabel('Temperature (°C)'); ax.set_xlabel('Date')
ax.set_title('Mediterranean basin-averaged daily temperature — Jan–Mar 2026')
ax.grid(alpha=0.3); ax.legend()
plt.tight_layout(); plt.show()

### 4. Temporal mean maps over the 90-day period

In [ ]:
mean_surf = thetao.mean(dim='time')
mean_bot  = ds_med['bottomT'].mean(dim='time')

fig, axes = plt.subplots(1, 2, figsize=(15, 4.5))
mean_surf.plot(ax=axes[0], cmap='turbo', cbar_kwargs={'label': '°C'})
axes[0].set_title('Mean surface temperature — Jan–Mar 2026')
mean_bot.plot(ax=axes[1], cmap='viridis', cbar_kwargs={'label': '°C'})
axes[1].set_title('Mean bottom temperature — Jan–Mar 2026')
for ax in axes:
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
plt.tight_layout(); plt.show()

### 5. Interactive map with hvplot (surface temperature, time slider)

In [ ]:
thetao.hvplot(
    x='longitude', y='latitude', groupby='time',
    rasterize=True, cmap='turbo', geo=True, tiles='OSM',
    clabel='°C', title='thetao (1 m) — daily'
)

---
## Salinity — same product, same period, same domain

**Product:** `MEDSEA_MULTIYEAR_PHY_006_004`  
**Dataset:** `cmems_mod_med_phy-sal_my_4.2km_P1D-m`  
**Variable:** `so` — sea water practical salinity (PSU) at 1 m depth  
**Period:** 2026-01-01 → 2026-03-31 | **Bbox:** Gulf of Tunisia (10°–10.9°E, 36.6°–37.2°N)

In [ ]:
import copernicusmarine

# Save credentials once to ~/.copernicusmarine so subsequent calls don't prompt
copernicusmarine.login(
    username="mazzabi1",
    password="Mazzabi@2019",   # ← replace with your actual password if this fails
    force_overwrite=True,
)

ds_sal = copernicusmarine.open_dataset(
    dataset_id="cmems_mod_med_phy-sal_my_4.2km_P1D-m",
    variables=["so"],
    minimum_longitude=10.0,
    maximum_longitude=10.9,
    minimum_latitude=36.6,
    maximum_latitude=37.2,
    start_datetime="2026-01-01T00:00:00",
    end_datetime="2026-03-31T00:00:00",
)
ds_sal

In [ ]:
import matplotlib.pyplot as plt

# Surface salinity (depth=0 = 1 m level)
so = ds_sal["so"].isel(depth=0)

# --- 1. Spatial map on first day ---
fig, ax = plt.subplots(figsize=(11, 4.5))
so.isel(time=0).plot(ax=ax, cmap="Blues_r", cbar_kwargs={"label": "PSU"})
ax.set_title(f"Surface salinity (so, 1 m) — {str(so.time.values[0])[:10]}")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.tight_layout(); plt.show()

In [ ]:
# --- 2. Basin-averaged daily salinity time series ---
ts_sal = so.mean(dim=["latitude", "longitude"])

fig, ax = plt.subplots(figsize=(11, 4))
ts_sal.plot(ax=ax, color="steelblue")
ax.set_ylabel("Salinity (PSU)"); ax.set_xlabel("Date")
ax.set_title("Basin-averaged daily surface salinity — Gulf of Tunisia, Jan–Mar 2026")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

In [ ]:
# --- 3. Temporal mean map ---
fig, ax = plt.subplots(figsize=(11, 4.5))
so.mean(dim="time").plot(ax=ax, cmap="Blues_r", cbar_kwargs={"label": "PSU"})
ax.set_title("Mean surface salinity — Gulf of Tunisia, Jan–Mar 2026")
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
plt.tight_layout(); plt.show()

In [ ]:
# --- 4. Interactive map with time slider ---
import hvplot.xarray  # noqa: F401

so.hvplot(
    x="longitude", y="latitude", groupby="time",
    rasterize=True, cmap="Blues_r", geo=True, tiles="OSM",
    clabel="PSU", title="Surface salinity (so, 1 m) — daily",
    width=700, height=500,
)